![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 24 -- Lab 1: Road Signs with a Detector

An Olympiad-style detection task: 877 street photos from Kaggle, each with an XML file listing its signs (traffic light, stop, speed limit, crosswalk) and their boxes, scored by mAP at IoU 0.5. You read the XML into tensors, write a Dataset that returns the target dictionary torchvision expects, take Faster R-CNN with its published COCO weights, replace its head for the four classes plus background, write the training loop with the loss dictionary and a validation mAP every epoch, look at the boxes it draws, and write a submission with one row per box.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** Kaggle `andrewmvd/road-sign-detection` (about 230 MB), downloaded by `kagglehub` in the setup cell: `images/roadN.png` (mostly 300 x 400) and `annotations/roadN.xml` in PASCAL VOC format. You split 700 photos for training and 177 for validation. Use a GPU (Runtime, Change runtime type, T4): an epoch takes under a minute there and several minutes on a CPU.

---
# Setup

In [ ]:
# torchmetrics computes mAP; pycocotools is its backend. Everything else is preinstalled on Colab.
!pip install -q torchmetrics pycocotools

In [ ]:
import os, glob
import xml.etree.ElementTree as ET
from collections import Counter
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import to_tensor
from torchvision.utils import draw_bounding_boxes
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_320_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchmetrics.detection import MeanAveragePrecision
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
path = kagglehub.dataset_download("andrewmvd/road-sign-detection")
IMG, ANN = os.path.join(path, "images"), os.path.join(path, "annotations")
CLASSES = ["trafficlight", "stop", "speedlimit", "crosswalk"]       # label 0 is reserved for background
ids = sorted(int(f[4:-4]) for f in os.listdir(IMG))                 # road0.png -> 0
print(len(ids), "photos;", open(os.path.join(ANN, "road0.xml")).read()[:420])

---
# Part 1 -- Boxes from XML into a Dataset

The *A Box Is Four Numbers* and *In Code: torchvision's Detection API* slides. VOC XML stores every object as a `name` and a `bndbox` with `xmin, ymin, xmax, ymax`: the corner format torchvision wants.

## Task 1: Parse one XML file

Write `parse_xml(xml_path)` that returns `(boxes, labels)`: for every `object` element, read the four corners from its `bndbox` into a row `[xmin, ymin, xmax, ymax]` and the class into `CLASSES.index(name) + 1` (so labels run 1 to 4). Return `torch.tensor(boxes, dtype=torch.float32)` of shape `(N, 4)` and `torch.tensor(labels, dtype=torch.int64)` of shape `(N,)`. Test it on `road0.xml`: one box `[98, 62, 208, 232]`, label 1.

Syntax hint (the shape of the call, not the answer):

```python
root = ET.parse(path).getroot()
for obj in root.findall("object"):
    name = obj.find("name").text
    x1 = float(obj.find("bndbox").find("xmin").text)
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert b0.shape == (1, 4) and b0.tolist() == [[98.0, 62.0, 208.0, 232.0]] and l0.tolist() == [1], 'parse_xml: one row per object, corners xmin, ymin, xmax, ymax; labels CLASSES.index(name) + 1'
print('Task 1 passed')

## Task 2: Count the classes and look at the boxes

Count how many boxes of each class the 877 files contain (a `Counter` over `parse_xml` of every file, mapped back to names) and print it. Then show 4 photos with their true boxes: `draw_bounding_boxes(image_uint8, boxes, labels=[names], colors="orange", width=3)` takes a `uint8` image tensor (`(to_tensor(img) * 255).to(torch.uint8)`) and returns one you can `permute(1, 2, 0)` for `imshow`. Which class dominates?

Syntax hint (the shape of the call, not the answer):

```python
drawn = draw_bounding_boxes(img_uint8, boxes, labels=list_of_names, colors="orange", width=3)   # (3, H, W) uint8
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert counts['speedlimit'] == 783 and sum(counts.values()) == 1244, 'counts: parse every XML file; 1,244 boxes in total, 783 of them speed limits'
print('Task 2 passed')

## Task 3: The Dataset class

Write `class SignDataset(Dataset)` with `__init__(self, ids)` storing the list, `__len__`, and `__getitem__(self, i)` that opens `images/road{id}.png` as RGB, turns it into a float tensor with `to_tensor`, parses the matching XML, and returns `(image, {"boxes": boxes, "labels": labels})`. Split `ids` with `torch.randperm(877, generator=torch.Generator().manual_seed(0))` into 700 training and 177 validation ids, build `train_set` and `val_set`, and print the shapes of `train_set[0]`.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(train_set) == 700 and len(val_set) == 177 and x0.dim() == 3 and t0['boxes'].shape[1] == 4, 'SignDataset: (3, H, W) float image and a dict with boxes (N, 4) and labels (N,)'
print('Task 3 passed')

Photos have different sizes and different numbers of boxes, so a batch cannot be stacked into one tensor. The loader below hands the model a *list* of images and a *list* of targets, which is exactly what torchvision's detectors take.

In [ ]:
def collate(batch):
    return tuple(zip(*batch))      # ([img, img, ...], [target, target, ...])

train_loader = DataLoader(train_set, batch_size=4, shuffle=True, collate_fn=collate)
val_loader = DataLoader(val_set, batch_size=4, collate_fn=collate)
images, targets = next(iter(train_loader))
print(len(images), images[0].shape, images[1].shape, targets[0]["labels"])

---
# Part 2 -- The detector

The *Two Families* and *In Code* slides: Faster R-CNN with a MobileNet backbone, its published COCO weights, and a new head for our five classes.

## Task 4: Load the detector and replace its head

Create `model = fasterrcnn_mobilenet_v3_large_320_fpn(weights="DEFAULT")`. Read `in_f = model.roi_heads.box_predictor.cls_score.in_features`, replace `model.roi_heads.box_predictor` with `FastRCNNPredictor(in_f, num_classes=5)`, and move the model to `device`. Print the new predictor and the number of parameters. Why 5 and not 4?

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert model.roi_heads.box_predictor.cls_score.out_features == 5, 'head: FastRCNNPredictor(in_f, num_classes=5), four classes plus background'
print('Task 4 passed')

## Task 5: Two modes, two outputs

In `model.train()` mode, pass the batch from the setup cell (`[im.to(device) for im in images]` and the targets moved to `device` with a dict comprehension) and print the keys and values of the returned loss dictionary. Then switch to `model.eval()`, pass the same images without targets inside `torch.no_grad()`, and print the keys of the first output and the shape of its `boxes`. The head is untrained, so the boxes mean nothing yet.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert set(loss_dict) >= {'loss_classifier', 'loss_box_reg'} and set(out[0]) >= {'boxes', 'labels', 'scores'}, 'train mode returns a dict of losses, eval mode a dict with boxes, labels, scores'
print('Task 5 passed')

---
# Part 3 -- Train, with mAP every epoch

The *Average Precision and mAP* slide. The metric object collects predictions and targets over the validation set and computes mAP at IoU 0.5; `map_50` is the number a competition would report.

## Task 6: The loop

Set `EPOCHS = 4`, `params = [p for p in model.parameters() if p.requires_grad]`, `opt = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=5e-4)` and `metric = MeanAveragePrecision(iou_thresholds=[0.5])`. For each epoch: `model.train()`, loop over `train_loader` moving images and targets to `device`, `loss = sum(model(imgs, tgts).values())`, the four moves, summing `loss.item() * len(imgs)`; then `model.eval()`, `metric.reset()`, and inside `torch.no_grad()` loop over `val_loader`, call `metric.update(preds_on_cpu, targets)` with the predictions moved to the CPU; `m50 = metric.compute()["map_50"].item()`; append `(train_loss, m50)` to `history`, save the weights to `best_signs.pt` when `m50` improves, print both numbers. Expected: mAP@0.5 around 0.7 after the first epoch and close to 0.8 by the fourth.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(history) == 4 and best > 0.4 and os.path.exists('best_signs.pt'), 'loop: 4 epochs of SGD on the summed loss dict, mAP@0.5 on val_loader each epoch, checkpoint on the best'
print('Task 6 passed')

## Task 7: Per-class AP

Load `best_signs.pt` into the model, build `metric_c = MeanAveragePrecision(iou_thresholds=[0.5], class_metrics=True)`, run the validation loop once more, and print `map_50` and the AP of each class (`result["map_per_class"]` in the order of `result["classes"]`, mapped to `CLASSES`). Which class is hardest, and what in the class counts of Task 2 explains it?

In [ ]:
# Your code here


---
# Part 4 -- Look, then submit

## Task 8: Draw the predictions

For the first 4 validation photos, run the model (eval, no_grad), keep the boxes with `scores > 0.5`, and draw them in purple next to the true boxes in orange (two rows of `draw_bounding_boxes`, labels as class names, the score in the label for the predictions). Where does it miss, and where does it draw two boxes for one sign?

In [ ]:
# Your code here


## Task 9: Write the submission

Run the model on every validation photo and collect one row per predicted box with score at least 0.3: `image` (`roadN.png`), `label` (class name), `x1, y1, x2, y2` (rounded to one decimal) and `score`. Save the rows as `submission.csv` without the index, read it back, and print its shape and first 5 rows. Every label must be one of the four class names.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
check = pd.read_csv('submission.csv')
assert set(check.columns) == {'image', 'label', 'x1', 'y1', 'x2', 'y2', 'score'} and set(check['label']) <= set(CLASSES) and len(check) > 100, 'submission: one row per box with score >= 0.3, class names as labels'
print('Task 9 passed')

---
## Task 10: Reflect

The detector reached a useful mAP after four epochs on 700 photos. Which part of it was already trained before you started, and which part did you add? Task 7 showed one class far behind the others. Name the two reasons the data suggests, and one move for each. A competition scored by mAP@0.5 rewards a box that overlaps the truth by half; mAP@[.5:.95] does not. Which of your predictions in Task 8 would lose most points under the stricter rule?

*Your answers here*

---
## Conclusion

- **Boxes from files**: VOC XML parsed into `(N, 4)` corner tensors and `(N,)` labels, drawn on the photos before any training.
- **A Dataset for detection**: images of different sizes and targets as dictionaries, batched as lists with a collate function.
- **A pretrained detector with a new head**: Faster R-CNN's loss dictionary in train mode, boxes with scores in eval mode, the Week 3 loop with one summed loss.
- **Scoring and hand-in**: mAP@0.5 every epoch and per class, predictions drawn against the truth, and a submission with one row per box.

---

Made By **Sattam Altwaim**